# Capítulo 5 · Algoritmo de Grover

## Objetivos

1. Comprender la amplificación de amplitud como mecanismo de aceleración cuántica.
2. Implementar el oráculo marcador y el operador de difusión de Grover.
3. Verificar la condición óptima de iteraciones $k \approx \pi/(4\arcsin(1/\sqrt{N}))$.
4. Analizar la ventaja cuadrática: $O(\sqrt{N})$ vs. $O(N)$ clásico.

---

## 5.1 Fundamento matemático

Sea $|s\rangle = H^{\otimes n}|0\rangle^{\otimes n}$ la superposición uniforme. El operador de Grover es:

$$G = D \cdot U_\omega, \quad D = 2|s\rangle\langle s| - I, \quad U_\omega|x\rangle = (-1)^{f(x)}|x\rangle$$

Geométricamente, $G$ realiza una rotación de ángulo $2\theta$ en el plano generado por $|s\rangle$ y $|\omega\rangle$, con $\sin\theta = 1/\sqrt{N}$. Tras $k$ iteraciones:

$$G^k|s\rangle = \sin((2k+1)\theta)|\omega\rangle + \cos((2k+1)\theta)|s'\rangle$$

La probabilidad de éxito se maximiza cuando $(2k+1)\theta \approx \pi/2$, es decir $k_{\text{opt}} \approx \pi/(4\theta)$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

print('Dependencias cargadas.')

## 5.2 Construcción del oráculo y el difusor

In [ ]:
def grover_oracle(n: int, target: int) -> QuantumCircuit:
    """Oráculo que marca el estado |target〉 con un cambio de fase -1.

    Implementación: multi-controlled Z (MCZ) con el estado target.

    Parámetros
    ----------
    n : int
        Número de qubits.
    target : int
        Índice del estado a marcar (0 ≤ target < 2^n).
    """
    qc = QuantumCircuit(n, name=f'Oracle({format(target, f"0{n}b")})')
    target_bits = format(target, f'0{n}b')

    # Negar los qubits que son '0' en el target (para que la MCZ dé -(-1))
    for i, bit in enumerate(reversed(target_bits)):
        if bit == '0':
            qc.x(i)

    # MCZ: implementada como H + MCX + H sobre el qubit más significativo
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)

    # Deshacer las negaciones
    for i, bit in enumerate(reversed(target_bits)):
        if bit == '0':
            qc.x(i)
    return qc


def grover_diffuser(n: int) -> QuantumCircuit:
    """Operador de difusión D = 2|s〉〈s| - I.

    Corresponde a una reflexión respecto al estado de superposición uniforme.
    """
    qc = QuantumCircuit(n, name='Diffuser')
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc


n = 4
target = 11  # = |1011〉
print('Oráculo:')
print(grover_oracle(n, target).draw('text'))
print('\nDifusor:')
print(grover_diffuser(n).draw('text'))

## 5.3 Circuito completo de Grover

In [ ]:
def grover_circuit(n: int, target: int, iterations: int | None = None) -> QuantumCircuit:
    """Construye el circuito completo del algoritmo de Grover.

    Parámetros
    ----------
    n : int
        Número de qubits.
    target : int
        Estado marcado.
    iterations : int, optional
        Número de iteraciones G. Por defecto, el óptimo teórico.
    """
    N = 2 ** n
    if iterations is None:
        theta = np.arcsin(1 / np.sqrt(N))
        iterations = int(np.round(np.pi / (4 * theta)))

    qc = QuantumCircuit(n, n)

    # Superposición inicial
    qc.h(range(n))
    qc.barrier()

    # k iteraciones de Grover
    oracle   = grover_oracle(n, target)
    diffuser = grover_diffuser(n)
    for _ in range(iterations):
        qc.compose(oracle,   inplace=True)
        qc.compose(diffuser, inplace=True)
        qc.barrier()

    # Medida
    qc.measure(range(n), range(n))
    return qc, iterations


n      = 4
target = 11
qc_grover, k_opt = grover_circuit(n, target)

print(f'n={n} qubits, N={2**n}, target={target} ({format(target, f"0{n}b")})')
print(f'Iteraciones óptimas: k = {k_opt}')

backend = AerSimulator()
job = backend.run(qc_grover, shots=4096)
counts = job.result().get_counts()

# Resultados
print('\nTop 5 resultados más frecuentes:')
for state, cnt in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    prob = cnt / 4096
    marker = ' ← TARGET' if int(state, 2) == target else ''
    print(f'  |{state}〉 ({int(state, 2):2d}): {cnt:4d} veces  ({prob:.3f}){marker}')

fig = QuantumVisualization.plot_histogram(
    counts, title=f'Grover: n={n}, target={format(target, f"0{n}b")} ({target}), k={k_opt}',
    color='#7ee787'
)
plt.show()

## 5.4 Evolución de la probabilidad de éxito con el número de iteraciones

In [ ]:
n = 4
target = 7
N = 2**n

# Cálculo analítico
k_range = range(1, 15)
theta = np.arcsin(1 / np.sqrt(N))
prob_analytic = [np.sin((2*k + 1) * theta)**2 for k in k_range]

# Simulación para cada k
prob_sim = []
backend = AerSimulator()
for k in k_range:
    qc, _ = grover_circuit(n, target, iterations=k)
    job = backend.run(qc, shots=2048)
    cnts = job.result().get_counts()
    target_str = format(target, f'0{n}b')
    prob_sim.append(cnts.get(target_str, 0) / 2048)

# Visualización
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_range), prob_analytic, '-', color='#58a6ff',
        linewidth=2, label='Analítico')
ax.plot(list(k_range), prob_sim, 'o', color='#f78166',
        markersize=7, label='Simulación Qiskit')
ax.axvline(int(np.round(np.pi / (4 * theta))),
           linestyle='--', color='#7ee787', alpha=0.7, label='k óptimo')
ax.set_xlabel('Iteraciones de Grover (k)')
ax.set_ylabel('Probabilidad de éxito P(target)')
ax.set_title(f'Probabilidad de éxito vs. iteraciones (n={n}, target={target})')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 5.5 Ejercicios propuestos

1. Implementa Grover para buscar **dos** elementos marcados simultáneamente. ¿Cómo cambia el número óptimo de iteraciones?

2. Para $n=2$, ejecuta el algoritmo para todas las posibles elecciones de target ($0,1,2,3$) y verifica el resultado.

3. Implementa el algoritmo de Grover para resolver el problema de 3-SAT con 3 variables. Codifica la cláusula $(x_1 \lor \neg x_2 \lor x_3)$ como oráculo.

4. ¿Qué ocurre si se aplican más de $k_{\text{opt}}$ iteraciones? Grafica la probabilidad de éxito para $k \in [1, 3k_{\text{opt}}]$.